### Import packages

In [ ]:
import numpy as np
import pandas as pd
from ethos_tised import SolarModel
import pathlib

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.optimize import minimize
from scipy.stats import ks_2samp
from sklearn.metrics import root_mean_squared_error
from sklearn.neighbors import KNeighborsClassifier
import seaborn as sns

### Load the hourly resolution data to downscale

In [ ]:
current_dir = pathlib.Path.cwd().parent
path_to_hourly_data = current_dir.joinpath(
    "missing_values", "hourly_data_missing_with_gaps.csv")

hourly_irrad_m = np.genfromtxt(
    path_to_hourly_data,
    delimiter=",",
)

### Calling the SolarModel of ETHOS.TISED to downscale the hourly data to one minute

#### This requires the latitude, longitude, date, and data as inputs

In [ ]:
synthetic = SolarModel(Lat=52.455778, Lon=13.523917, date=2018, data=hourly_irrad_m)

### Load the measured data

In [ ]:
path_to_measured_data = current_dir /'Results'/'Berlin'/'Berlin2018.csv'

data = pd.read_csv(path_to_measured_data)
data.describe()

### Finding the annual NRMSE

In [ ]:
rms = root_mean_squared_error(data["ghi"], synthetic["synthetic_ghi"])
nrmse = round(rms / (data["ghi"].max() - data["ghi"].min()) * 100, 2)
print("###############################################################################")
print("Synthetic GHI")
print("###############################################################################")
print("Minute RMSE is:", round(rms, 2))
print("Minute NRMSE is:", nrmse, "%")

### Calling the SolarModel of ETHOS.TISED to downscale the hourly data to one minute

#### This requires the latitude, longitude, date, and data as inputs

Normally, calling `SolarModel(...)` returns only the final minute-resolution `synthetic` DataFrame — the model instance itself, along with the intermediate hourly-imputation results, is discarded as soon as construction finishes.

To also inspect what the gap-filling step did to the hourly input, the cell below builds the instance directly with `object.__new__(SolarModel)` followed by `model.__init__(...)`, instead of calling `SolarModel(...)`. This runs the exact same pipeline (timezone/altitude resolution, missing-value imputation, downscaling), but keeps a handle to `model` afterwards, exposing:

- `model.synthetic` — the same minute-resolution output a normal call would return (assigned to `synthetic` below, so the rest of the notebook is unaffected)
- `model.hourly_irrad_m` — the post-imputation hourly `[day, hour, ghi]` array
- `model.imputation_method` — a per-hour label (`observed`, `night`, `previous_day`, `interpolation`, `knn`) recording which rule in `ethos_tised.imputation.impute_hourly_ghi` filled each hour

`imputed_hourly` combines the last two into a single table, so gaps and how they were filled can be reviewed hour by hour.

In [ ]:
# Build the instance directly instead of letting __new__ hand back only `.synthetic`
model = object.__new__(SolarModel)
model.__init__(Lat=52.455778, Lon=13.523917, Altitude=None, date=2018, data=hourly_irrad_m)

# Same minute-resolution result a normal `SolarModel(...)` call would return
synthetic = model.synthetic

# Table of every hour: filled value + which rule filled it
imputed_hourly = pd.DataFrame({
    "ghi": model.hourly_irrad_m[:, 2],
    "method": model.imputation_method.values,
}, index=model.imputation_method.index)

imputed_hourly.head()

In [ ]:
imputed_hourly